
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# PII Lookup Table

This notebook allows you to programmatically:

* Generate the DLT pipeline
* Trigger a pipeline run
* Explore the resultant DAG
* Land a new batch of data

## The Pipeline
The pipeline we are using in this lesson is located [here]($./Pipeline/ADE 3.1.1 - PII Lookup Table).

Run the following cell to configure your working environment for this course.

In [0]:
%run ./Includes/Classroom-Setup-03.1

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
%python
display(dbutils.fs.ls("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"))

path,name,size,modificationTime
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/bronze/,bronze/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/date-lookup/,date-lookup/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/,ecommerce/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/flights/,flights/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/gym-logs/,gym-logs/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/gym-mac-logs/,gym-mac-logs/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/healthcare/,healthcare/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/kafka-30min/,kafka-30min/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/movie_ratings/,movie_ratings/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/nyctaxi-with-zipcodes/,nyctaxi-with-zipcodes/,0,1743714329668


In [0]:
%python
display(dbutils.fs.ls("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"))

path,name,size,modificationTime
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/bronze/,bronze/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/date-lookup/,date-lookup/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/,ecommerce/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/flights/,flights/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/gym-logs/,gym-logs/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/gym-mac-logs/,gym-mac-logs/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/healthcare/,healthcare/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/kafka-30min/,kafka-30min/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/movie_ratings/,movie_ratings/,0,1743714329668
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/nyctaxi-with-zipcodes/,nyctaxi-with-zipcodes/,0,1743714329668


Resetting the learning environment (pii):
| removing the working directory "dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/advanced-data-engineering-with-databricks/pii"...(0 seconds)

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"

Validating the locally installed datasets:
| listing local files...(8 seconds)
| validation completed...(8 seconds total)

Creating & using the schema "shifajamali55_soeb_da_adewd_pii" in the catalog "hive_metastore"...(7 seconds)
Cloning the "shifajamali55_soeb_da_adewd_pii_lookup.date_lookup" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/date-lookup"....(54 seconds)
Cloning the "shifajamali55_soeb_da_adewd_pii_lookup.user_lookup" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-lookup"....(25 seconds)


In [0]:
%python
# List the files in the directory
display(dbutils.fs.ls("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg"))

# Load all JSON files into a single DataFrame
df = spark.read.json("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg")

# Display the DataFrame
display(df)

# Calculate the size of the dataset uncompressed
uncompressed_size = df.rdd.map(lambda row: len(str(row))).sum()
print(f"Uncompressed size of the dataset: {uncompressed_size} bytes")

path,name,size,modificationTime
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/11745.json,11745.json,110,1739720997000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/12140.json,12140.json,108,1739720997000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/12227.json,12227.json,110,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/12474.json,12474.json,110,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/13559.json,13559.json,108,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/13937.json,13937.json,110,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/14232.json,14232.json,108,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/14508.json,14508.json,108,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/14633.json,14633.json,110,1739720998000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg/15149.json,15149.json,108,1739720999000


device_id,mac_address,registration_timestamp,user_id
190960,14:cd:d6:db:70:f6,1.575763595E9,11745
114131,57:24:ac:8c:75:ea,1.575816942E9,12227
118440,4c:c5:9f:cb:13:bd,1.575178082E9,12474
143442,dd:96:be:e9:1e:f4,1.575399817E9,13937
175406,1d:69:69:75:d0:aa,1.57609761E9,14633
144334,cd:53:ec:b5:60:49,1.57528087E9,16819
117058,08:87:1c:9c:c7:a9,1.576412429E9,17217
112008,c8:45:8f:77:34:24,1.576046918E9,17807
178310,28:82:3d:8c:d6:f8,1.575388301E9,18587
163871,88:6c:c1:66:9b:cc,1.575223206E9,19005


Uncompressed size of the dataset: 10600 bytes



** WARNING: POLICY NOT FOUND ***********************************************************************
* Could not find the cluster policy "DBAcademy DLT".
* Please run the notebook Includes/Workspace-Setup to address this error.
****************************************************************************************************

Loading batch #1 to the stream...Loaded 37 records

Predefined tables in "shifajamali55_soeb_da_adewd_pii":
| __apply_changes_storage_users
| bronze
| date_lookup
| registered_users
| user_lookup
| users
| users_bronze
| users_cdc_clean

Predefined paths variables:
| DA.paths.working_dir:      dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/advanced-data-engineering-with-databricks/pii
| DA.paths.user_db:          dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/advanced-data-engineering-with-databricks/pii/database.db
| DA.paths.datasets:         dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04
| DA.paths.stream_source:    dbfs:/mnt/dbacademy-us

In [0]:
%python
# List the files in the directory
files = dbutils.fs.ls("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/user-reg")

# Calculate the total size of the dataset
total_size = sum(file.size for file in files)
print(f"Total size of the dataset: {total_size} bytes")

Total size of the dataset: 10926 bytes


In [0]:
total_rows = df.count()
print(f"Total number of rows: {total_rows}")

Total number of rows: 2703


In [0]:
%python
# List the files in the directory
display(dbutils.fs.ls("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw"))

# Load all JSON files into a single DataFrame
df = spark.read.format("delta").load("dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw")

# Display the DataFrame
display(df)

# Calculate the size of the dataset uncompressed
uncompressed_size = df.rdd.map(lambda row: len(str(row))).sum()
print(f"Uncompressed size of the dataset: {uncompressed_size} bytes")

path,name,size,modificationTime
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/_delta_log/,_delta_log/,0,1743718096400
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00000-a32ba7f5-b322-4df9-bceb-e6e94f393519-c000.snappy.parquet,part-00000-a32ba7f5-b322-4df9-bceb-e6e94f393519-c000.snappy.parquet,21018,1739720956000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00000-d693df4c-138c-4527-a535-246d2e362ab6-c000.snappy.parquet,part-00000-d693df4c-138c-4527-a535-246d2e362ab6-c000.snappy.parquet,21339,1739720956000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00000-ea64f1dc-d1c5-421a-a799-e1f37bd634ed-c000.snappy.parquet,part-00000-ea64f1dc-d1c5-421a-a799-e1f37bd634ed-c000.snappy.parquet,22488,1739720956000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00001-233c0c13-815e-4b8b-a788-75b67dbfaa11-c000.snappy.parquet,part-00001-233c0c13-815e-4b8b-a788-75b67dbfaa11-c000.snappy.parquet,20427,1739720956000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00001-6a0244ad-8d92-46b9-980e-fd228be85f22-c000.snappy.parquet,part-00001-6a0244ad-8d92-46b9-980e-fd228be85f22-c000.snappy.parquet,17615,1739720957000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00001-ebd30bc2-721f-4a83-a6dc-d1b75d8d0342-c000.snappy.parquet,part-00001-ebd30bc2-721f-4a83-a6dc-d1b75d8d0342-c000.snappy.parquet,18239,1739720957000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00002-658faa50-5a59-47da-9da6-91ff934adaa2-c000.snappy.parquet,part-00002-658faa50-5a59-47da-9da6-91ff934adaa2-c000.snappy.parquet,16870,1739720957000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00002-7419ba0b-127f-421a-9814-63bb90155ce7-c000.snappy.parquet,part-00002-7419ba0b-127f-421a-9814-63bb90155ce7-c000.snappy.parquet,15770,1739720957000
dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/pii/raw/part-00002-f34d72a1-810f-486a-ab0c-c0d9436e63c3-c000.snappy.parquet,part-00002-f34d72a1-810f-486a-ab0c-c0d9436e63c3-c000.snappy.parquet,17942,1739720957000


mrn,dob,sex,gender,first_name,last_name,street_address,zip,city,state,updated,batch
20390858,2000-02-11,F,F,Melanie,Parsons,2494 Suzanne Mews Suite 056,91204,Glendale,CA,2020-01-20T13:16:56.853Z,2
50524996,1951-11-26,M,M,Jesse,Duncan,266 Collins Road Apt. 486,90254,Hermosa Beach,CA,2020-01-20T13:45:54.707Z,2
61389274,1964-07-29,M,M,Dylan,Mills,8377 Bernard Crescent Apt. 091,90230,Culver City,CA,2020-01-20T13:54:29.947Z,2
71535493,1935-12-20,F,F,Lori,Small,309 Susan Unions Apt. 756,90601,Whittier,CA,2020-01-20T11:39:25.313Z,2
93378187,1980-11-21,F,F,Sherry,Martinez,5693 Dale Lodge Apt. 011,90220,Compton,CA,2020-01-20T11:49:13.153Z,2
80319715,1935-03-22,F,F,Samantha,Jimenez,799 Jeffery Wells,91007,Arcadia,CA,2020-01-05T07:32:08.675Z,2
19021993,1987-03-18,M,M,Cody,Phillips,6165 Lowery Pines Apt. 962,91308,West Hills,CA,2020-01-05T07:50:07.798Z,2
23931323,1955-04-07,F,F,Tracy,Hamilton,88357 Ruben Mountain,90012,Los Angeles,CA,2020-01-20T19:39:54.993Z,2
89082924,1941-10-08,M,M,David,Henderson,4880 Nathan Viaduct Suite 255,90032,Los Angeles,CA,2020-01-11T08:48:04.528Z,2
10267471,1937-11-04,M,M,Michael,Mitchell,16248 Matthews Keys,91311,Chatsworth,CA,2020-01-23T11:55:16.02Z,2


Uncompressed size of the dataset: 709726 bytes



## Generate DLT Pipeline
Run the cell below to auto-generate your DLT pipeline using the provided configuration values.

Once the pipeline is ready, a link will be provided to navigate you to your auto-generated pipeline in the Pipeline UI.

In [0]:
DA.generate_pipeline()

Pipeline Name:,
Storage Location:,
Target:,
Policy:,
source:,
lookup_db:,
Notebook #1 Path:,


Created the pipeline "shifajamali55-soeb-da-adewd-pii: Pipeline - pii": 27db9976-11fe-417e-a703-8019361d9604

'27db9976-11fe-417e-a703-8019361d9604'

## Trigger Pipeline Run

With a pipeline created, you will now run the pipeline. The initial run will take several minutes while a cluster is provisioned. Subsequent runs will be appreciably quicker.

Explore the DAG - As the pipeline completes, the execution flow is graphed. With each triggered update, all newly arriving data will be processed through your pipeline. Metrics will always be reported for current run.

In [0]:
DA.start_pipeline()

Current state is CREATED, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES, sleeping 15 seconds.
Current state is WAITING_FOR_RESOURCES

'43f248f5-ecb6-4169-b6d0-cbb2a3cfae1c'

## Land New Data

Run the cell below to land more data in the source directory, then manually trigger a pipeline update.

As we continue through the course, you can return to this notebook and use the method provided below to land new data. Running this entire notebook again will delete the underlying data files for both the source data and your DLT Pipeline.

In [0]:
DA.user_reg_stream.load()

Loading batch #2 to the stream...Loaded 5 records


## Process All Remaining Data
To continuously load all remaining batches of data to the source directory, call the same load method above with the **`continuous`** parameter set to **`True`**.

Trigger another update to process the remaining data.

In [0]:
DA.user_reg_stream.load(continuous=True)  # Load all remaining batches of data
DA.start_pipeline()  # Trigger another pipeline update

Loading all batches to the stream...Loaded 58 records
Current state is CREATED, sleeping 15 seconds.
Current state is RUNNING, sleeping 15 seconds.
The final state is COMPLETED.


'88b622f2-720a-4e09-bb1f-11498a3cb739'


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>